# notebook for trying a few ideas

In [1]:
import numpy as np
import wandb
import pandas as pd
import torch
import torch.nn as nn
import os

# Metrics and testing
from sklearn.metrics import roc_auc_score, accuracy_score, average_precision_score
from sklearn.model_selection import train_test_split

# My utils and setup
from nn.lstmgru_mlp import LSTMGRU_MLP
from utils.dataset import BinaryDataset
from utils.utils import Utils
from utils.loader import Loader
from utils.visualizers import Visualizer
from torch.utils.data import DataLoader
from utils.cosine import generate_cosine_pattern_graphs

# Import all embedding methods
from utils.embedding_methods.betweenness import EmbedBetweenness
from utils.embedding_methods.closeness import EmbedCloseness
from utils.embedding_methods.degree import EmbedDegree
from utils.embedding_methods.forman_ricci import EmbedForman
from utils.embedding_methods.weight import EmbedWeight

In [ ]:
df = pd.read_csv('data/input/raw/labels/networkaeternity_Label.csv')
values = df.values

In [ ]:
seed = 42  # For training and consistency
my_utils = Utils()
my_utils.set_seeds(seed)

# Parameters for graph generation
num_graphs = 250 # Number of graphs for training
max_nodes = 150  # Maximum number of nodes
avg_edges_per_node = 10  # Average number of edges per node
period_days = 30  # Set the period of the cosine cycle
start_size = 10  # Starting size for nodes
max_weight = 1  # Max weight of an edge

num_buckets = 10

# Generate graphs with cosine patterns for nodes and random edges
synthetic_graphs, labels = generate_cosine_pattern_graphs(num_graphs, max_nodes, avg_edges_per_node, period_days, start_size, max_weight)

DELETE LATER

In [ ]:
from utils.loader import Loader
from utils.visualizers import Visualizer

my_loader = Loader()
dataset = 'networkaragon'
activation_name = 'Forman'
my_visualizer = Visualizer(dataset='aragon', task='Binary')
data, labels = my_loader.load_data(dataset, activation_name)  # Load embeddings and labels

embedding = data[0]

my_visualizer.display_embeddings(data, data, data)

## Test Embedding Methods on Cosine Graphs

In [4]:
# Set up the embedding methods for testing
activations = [EmbedBetweenness, EmbedCloseness, EmbedDegree, EmbedForman, EmbedWeight]
activation_names = ['Betweenness', 'Closeness', 'Degree', 'Forman_Weight_1', 'Weight']
datasets = ['networkaeternity', 'networkcoindash', 'networkadex', 'networkbancor', 'CollegeMsg',  'Reddit_B']

In [5]:
num_layers = [3, 2]
dropouts = [0, 0.2]
hidden_dim_1 = [64, 128, 256]
hidden_dim_2 = [32, 64, 128, 256]
mlp_dims = [32, 64]
learning_rates = [0.0001, 0.001]
l2_regularizations = [0.00001, 0.0001, 0.001]
num_epochs = 500

top_runs = {}  # Store the best models for the best runs as a tuple of (run_id, model, valid_aucroc)

# Constants
output_dim = 1  # Binary classification
input_dim = 30  # 30-dimensional embeddings
patience = 25  # Early stopping patience

csv_file_path = 'data/output/results/BinaryTesting/data/cosine_gs.csv'

# Write the header if the file doesn't already exist
if not os.path.isfile(csv_file_path):
    pd.DataFrame(columns=['run_id', 'activation', 'seed', 'hidden_dim_1', 'hidden_dim_2', 'mlp_dim', 'learning_rate', 'dropout', 'l2_regularization', 'num_layers_LSTM', 'num_layers_GRU', 'trained_epochs', 'train_loss', 'valid_loss', 'train_aucroc', 'valid_aucroc', 'train_aucpr', 'valid_aucpr', 'train_accuracy', 'valid_accuracy']).to_csv(csv_file_path, index=False)

In [ ]:
# Check distribution of labels
percent_1s = (labels.count(1) / len(labels) ) * 100
percent_0s = (labels.count(0) / len(labels) ) * 100
print(f'The labels have {percent_1s:.2f}% 1\'s and {percent_0s:.2f}% 0\'s')
print(f'The labels have {labels.count(1)} 1\'s and {labels.count(0)} 0\'s')

In [7]:
def update_top_models(top_runs, activation, run_id, model, valid_aucroc, top_x=3):
    # If not initialized, add it
    if activation not in top_runs:
        top_runs[activation] = []
    
    top_runs[activation].append({'model': model, 'run_id': run_id, 'valid_aucroc': valid_aucroc})  # Add to the list
    top_runs[activation].sort(key=lambda x: x['valid_aucroc'], reverse=True)  # Sort the list by score (descending)
    
    # Keep only the top x models
    if len(top_runs[activation]) > top_x:
        top_runs[activation].pop()
    
    return top_runs

Regression Testing

In [ ]:
for activation, activation_name in zip(activations, activation_names):
    
    # Only testing Forman-Ricci right now
    if activation != EmbedForman:
        continue 
    
    my_activation = activation(num_buckets=num_buckets)    
    # Since Forman Ricci requires directed edges
    if activation==EmbedForman:
        embeddings = my_activation.process_graphs_for_embeddings(synthetic_graphs, is_directed=False)
    else:
        embeddings = my_activation.process_graphs_for_embeddings(synthetic_graphs)
    
    wandb.init(
        project="embedding_cosine_gs", 
        name=f"{activation_name}_buckets{num_buckets}", 
        reinit=True
    )
    
    # Split data 70/15/15
    X_train, X_tmp, y_train, y_tmp = train_test_split(embeddings, labels, test_size=0.3, shuffle=False)
    X_val, X_test, y_val, y_test = train_test_split(X_tmp, y_tmp, test_size=0.5, shuffle=False)

    train_dataset = BinaryDataset(X_train, y_train)
    valid_dataset = BinaryDataset(X_val, y_val)
    test_dataset = BinaryDataset(X_test, y_test)
    train_loader = DataLoader(train_dataset, batch_size=16, shuffle=False, drop_last=False)  # On specific datasets the shape mismatch will crash the code on some batches
    valid_loader = DataLoader(valid_dataset, batch_size=16, shuffle=False, drop_last=False)  # drop_last fixes this issue
    test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False, drop_last=False)
    best_valid_aucroc = float('-inf')  # Init
    
    #split_index = len(train_loader) + len(valid_loader)
    for num_layer in num_layers:
        for dropout in dropouts:
            for hidden_1 in hidden_dim_1:
                for hidden_2 in hidden_dim_2:
                    for mlp_dim in mlp_dims:
                        for lr_val in learning_rates:  
                            for l2_val in l2_regularizations:
                                curr_batch_best_aucroc = float('-inf')  # Init
                    
                                # Initialize wandb
                                run = wandb.init(project="embedding_cosine_gs", config={
                                    'activation': activation_name,
                                    'num_layers': num_layer,
                                    'dropout': dropout,
                                    'l2_regularization': l2_val,
                                    'hidden_dim_1': hidden_1,
                                    'hidden_dim_2': hidden_2,
                                    'mlp_dim': mlp_dim,
                                    'learning_rate': lr_val,
                                    'seed': seed
                                })
                                
                                
                                # Setup
                                no_improvement_counter = 0  # Number of epochs that we haven't seen an improvement in the validation AUCROC
                                model = LSTMGRU_MLP(input_dim, output_dim, hidden_dim_1=hidden_1, hidden_dim_2=hidden_2, mlp_dim=mlp_dim, dropout=dropout, num_layers_LSTM=num_layer, num_layers_GRU=num_layer)
                                optimizer = torch.optim.Adam(model.parameters(), lr=lr_val, weight_decay=l2_val)
                                criterion = nn.BCELoss()    
                                
                                for epoch in range(num_epochs):
                                    model.train()
                                    train_loss, train_aucroc, train_aucpr, train_accuracy = model.train_model_binary(model, train_loader, optimizer, criterion)

                                    with torch.no_grad():
                                        model.eval()
                                        val_preds = []
                                        valid_loss = 0
                                        for x, y in valid_loader:
                                            output = model(x)  # Maintain hidden state across time steps
                                            y = y.squeeze().float()
                                            loss = criterion(output, y)
                                            valid_loss += loss.item()
                                            val_preds.append(output.detach().numpy())

                                        # Compute metrics
                                        valid_loss /= len(valid_loader)
                                        val_preds = np.concatenate(val_preds, axis=0)  # Ensure val_preds is a flat array
                                        val_preds = np.array(val_preds)
                                        valid_aucroc = roc_auc_score(y_val, val_preds)
                                        valid_aucpr = average_precision_score(y_val, val_preds)
                                        val_pred_labels = [1 if prob >= 0.5 else 0 for prob in val_preds]  # Since accuracy requires labels
                                        valid_accuracy = accuracy_score(y_val, val_pred_labels)
                                        
                                    # Log each epoch results
                                    wandb.log({
                                        'epoch': epoch,
                                        'train_loss': train_loss,
                                        'valid_loss': valid_loss,
                                        'train_aucroc': train_aucroc,
                                        'valid_aucroc': valid_aucroc,
                                        'train_aucpr': train_aucpr,
                                        'valid_aucpr': valid_aucpr,
                                        'train_accuracy': train_accuracy,
                                        'valid_accuracy': valid_accuracy
                                    })

                                    # Optimize for the best aucroc
                                    if valid_aucroc >= curr_batch_best_aucroc:
                                        # Save for dataframe
                                        best_moment_row = {
                                            'run_id': run.name,  # For checking Wandb Logs
                                            'activation': activation_name,
                                            'seed': seed,
                                            'hidden_dim_1': hidden_1,
                                            'hidden_dim_2': hidden_2,
                                            'mlp_dim': mlp_dim,
                                            'learning_rate': lr_val,
                                            'dropout': dropout,
                                            'l2_regularization': l2_val,
                                            'num_layers_LSTM': num_layer,
                                            'num_layers_GRU': num_layer,
                                            'trained_epochs': epoch + 1,
                                            'train_loss': train_loss,
                                            'valid_loss': valid_loss,
                                            'train_aucroc': train_aucroc,
                                            'valid_aucroc': valid_aucroc,
                                            'train_aucpr': train_aucpr,
                                            'valid_aucpr': valid_aucpr,
                                            'train_accuracy': train_accuracy,
                                            'valid_accuracy': valid_accuracy
                                        }
                                        
                                        # Save the model
                                        top_runs = update_top_models(top_runs, activation, run.name, model, valid_aucroc, top_x=3)
                                        curr_batch_best_aucroc = valid_aucroc
                                        
                                        # If we have a new best model for this activation
                                        if valid_aucroc > best_valid_aucroc:
                                            best_valid_aucroc = valid_aucroc
                                            print(f'We have a new best model with a Validation AUCROC: {valid_aucroc}')
                                            test_loss, test_aucroc, test_aucpr, test_accuracy = model.test_model_binary(model, test_loader, criterion, y_test)
                                            print(f"""\tTest Loss: {valid_loss}\n\tTest Validation AUCROC: {test_aucroc}\n\tTest AUCPR: {test_aucpr}\n\tTest Accuracy: {test_accuracy}\n
                                            """)
                                    
                                    # Early stopping only after 50 epochs
                                    if epoch >= 50:
                                        if valid_aucroc >= curr_batch_best_aucroc:
                                            no_improvement_counter = 0
                                            curr_batch_best_aucroc = valid_aucroc
                                        else:
                                            no_improvement_counter += 1
                                            
                                        if no_improvement_counter == patience:
                                            print(f'Training ending at epoch number: {epoch + 1}')
                                            break
                                        
                                    # Display current results
                                    if epoch % 5 - 4 == 0:
                                        print(f"""
                                            Epoch {epoch+1}/{num_epochs}:\n\tTrain Loss: {train_loss}, Validation Loss: {valid_loss}\n\tTrain AUCROC: {train_aucroc}, Validation AUCROC: {valid_aucroc}\n\tTrain AUCPR: {train_aucpr}, Validation AUCPR: {valid_aucpr}\n\tTrain Accuracy: {train_accuracy}, Validation Accuracy: {valid_accuracy}\n
                                        """)

                                # Save the best moment from this training
                                pd.DataFrame([best_moment_row]).to_csv(csv_file_path, mode='a', header=False, index=False)
                                        
    wandb.finish()  # Close run

See summary of best runs

In [ ]:
import pandas as pd
csv_file_path = 'data/output/results/BinaryTesting/data/embedding_testing.csv'

df = pd.read_csv(csv_file_path)

tmp_df = df[df['activation'] == 'Weight']  # If you want to filter by a certain activation

pd.set_option('display.max_columns', None)  # Show all columns
pd.set_option('display.width', 1000)        # Set the maximum width for the display
pd.set_option('display.max_colwidth', None) # Show full column content if truncated

top_5_loss = df.nsmallest(6, 'valid_loss')

# Top 5 based on highest valid_aucroc (descending order)
top_5_aucroc = df.nlargest(6, 'valid_aucroc')

# Top 5 based on highest valid_accuracy (descending order)
top_5_accuracy = df.nlargest(6, 'valid_accuracy')

# Top 5 based on highest valid_aucpr (descending order)
top_5_aucpr = df.nlargest(6, 'valid_aucpr')

# Print the results
print("Top 10 rows based on lowest valid_loss:")
print(top_5_loss)

print("\nTop 10 rows based on highest valid_aucroc:")
print(top_5_aucroc)

print("\nTop 10 rows based on highest valid_accuracy:")
print(top_5_accuracy)

print("\nTop 10 rows based on highest valid_aucpr:")
print(top_5_aucpr)# Compute averages, stdev, metrics

## Test the top 3 models from each activation

In [9]:
# Save the models first
for activation, info in top_runs.items():
    for rank, entry in enumerate(info, start=1):  # To get the model ranking
        
        # Since I did the naming wrong
        if activation == EmbedBetweenness:
            activation_name = "Betweenness"
        elif activation == EmbedCloseness:
            activation_name = "Closeness"
        elif activation == EmbedDegree:
            activation_name = "Degree"
        elif activation == EmbedForman:
            activation_name = "FormanRicci"
        elif activation == EmbedWeight:
            activation_name = "Weight"
        run_name = entry['run_id'] 
        model = entry['model']  
        
        # Run name is here for clarity and reference
        torch.save(model.state_dict(), f"data/output/cached_model/BinaryTesting/{run_name}_{activation_name}_{rank}.pt")

Set up embeddings

In [10]:
my_activation = EmbedCloseness(num_buckets=num_buckets)   
closeness_embeddings = my_activation.process_graphs_for_embeddings(synthetic_graphs)
my_activation = EmbedDegree(num_buckets=num_buckets)   
degree_embeddings = my_activation.process_graphs_for_embeddings(synthetic_graphs)
my_activation = EmbedBetweenness(num_buckets=num_buckets)   
betweenness_embeddings = my_activation.process_graphs_for_embeddings(synthetic_graphs)
my_activation = EmbedForman(num_buckets=num_buckets)   
forman_embedddings = my_activation.process_graphs_for_embeddings(synthetic_graphs, is_directed=False)
my_activation = EmbedWeight(num_buckets=num_buckets)   
weight_embeddings = my_activation.process_graphs_for_embeddings(synthetic_graphs)

X_train_closeness, X_tmp_closeness, y_train_closeness, y_tmp_closeness = train_test_split(closeness_embeddings, labels, test_size=0.3, shuffle=False)
X_val_closeness, X_test_closeness, y_val_closeness, y_test_closeness = train_test_split(X_tmp_closeness, y_tmp_closeness, test_size=0.5, shuffle=False)
train_dataset = BinaryDataset(X_train_closeness, y_train_closeness)
valid_dataset = BinaryDataset(X_val_closeness, y_val_closeness)
test_dataset = BinaryDataset(X_test_closeness, y_test_closeness)
train_closeness = DataLoader(train_dataset, batch_size=16, shuffle=False, drop_last=False)  # On specific datasets the shape mismatch will crash the code on some batches
valid_closeness = DataLoader(valid_dataset, batch_size=16, shuffle=False, drop_last=False)  # drop_last fixes this issue
test_closeness = DataLoader(test_dataset, batch_size=16, shuffle=False, drop_last=False)

X_train_degree, X_tmp_degree, y_train_degree, y_tmp_degree = train_test_split(degree_embeddings, labels, test_size=0.3, shuffle=False)
X_val_degree, X_test_degree, y_val_degree, y_test_degree = train_test_split(X_tmp_degree, y_tmp_degree, test_size=0.5, shuffle=False)
train_dataset = BinaryDataset(X_train_degree, y_train_degree)
valid_dataset = BinaryDataset(X_val_degree, y_val_degree)
test_dataset = BinaryDataset(X_test_degree, y_test_degree)
train_degree = DataLoader(train_dataset, batch_size=16, shuffle=False, drop_last=False)  # On specific datasets the shape mismatch will crash the code on some batches
valid_degree = DataLoader(valid_dataset, batch_size=16, shuffle=False, drop_last=False)  # drop_last fixes this issue
test_degree = DataLoader(test_dataset, batch_size=16, shuffle=False, drop_last=False)

X_train_betweenness, X_tmp_betweenness, y_train_betweenness, y_tmp_betweenness = train_test_split(betweenness_embeddings, labels, test_size=0.3, shuffle=False)
X_val_betweenness, X_test_betweenness, y_val_betweenness, y_test_betweenness = train_test_split(X_tmp_betweenness, y_tmp_betweenness, test_size=0.5, shuffle=False)
train_dataset = BinaryDataset(X_train_betweenness, y_train_betweenness)
valid_dataset = BinaryDataset(X_val_betweenness, y_val_betweenness)
test_dataset = BinaryDataset(X_test_betweenness, y_test_betweenness)
train_betweenness = DataLoader(train_dataset, batch_size=16, shuffle=False, drop_last=False)  # On specific datasets the shape mismatch will crash the code on some batches
valid_betweenness = DataLoader(valid_dataset, batch_size=16, shuffle=False, drop_last=False)  # drop_last fixes this issue
test_betweenness = DataLoader(test_dataset, batch_size=16, shuffle=False, drop_last=False)

X_train_forman, X_tmp_forman, y_train_forman, y_tmp_forman = train_test_split(forman_embedddings, labels, test_size=0.3, shuffle=False)
X_val_forman, X_test_forman, y_val_forman, y_test_forman = train_test_split(X_tmp_forman, y_tmp_forman, test_size=0.5, shuffle=False)
train_dataset = BinaryDataset(X_train_forman, y_train_forman)
valid_dataset = BinaryDataset(X_val_forman, y_val_forman)
test_dataset = BinaryDataset(X_test_forman, y_test_forman)
train_forman = DataLoader(train_dataset, batch_size=16, shuffle=False, drop_last=False)  # On specific datasets the shape mismatch will crash the code on some batches
valid_forman = DataLoader(valid_dataset, batch_size=16, shuffle=False, drop_last=False)  # drop_last fixes this issue
test_forman = DataLoader(test_dataset, batch_size=16, shuffle=False, drop_last=False)

X_train_weight, X_tmp_weight, y_train_weight, y_tmp_weight = train_test_split(weight_embeddings, labels, test_size=0.3, shuffle=False)
X_val_weight, X_test_weight, y_val_weight, y_test_weight = train_test_split(X_tmp_closeness, y_tmp_closeness, test_size=0.5, shuffle=False)
train_dataset = BinaryDataset(X_train_weight, y_train_weight)
valid_dataset = BinaryDataset(X_val_weight, y_val_weight)
test_dataset = BinaryDataset(X_test_weight, y_test_weight)
train_weight = DataLoader(train_dataset, batch_size=16, shuffle=False, drop_last=False)  # On specific datasets the shape mismatch will crash the code on some batches
valid_weight = DataLoader(valid_dataset, batch_size=16, shuffle=False, drop_last=False)  # drop_last fixes this issue
test_weight = DataLoader(test_dataset, batch_size=16, shuffle=False, drop_last=False)

Test on Test Split

In [ ]:
# Test on these models
path = 'data/output/cached_model/BinaryTesting'
model_files = [f for f in os.listdir(path) if f.endswith(".pt") and os.path.isfile(os.path.join(path, f))]


models = {}
for model_file in model_files:
    file_path = os.path.join(path, model_file)
    model_name = os.path.splitext(model_file)[0]  # Use file name without extension as key
    models[model_name] = torch.load(file_path)

for model_name, state in models.items():
    run = model_name.split('_')[0]
    row = df[df['run_id'] == run]
    
    # Get hyperparameters:
    hidden_1 = int(row['hidden_dim_1'].values[0])
    hidden_2 = int(row['hidden_dim_2'].values[0])
    mlp_dim = int(row['mlp_dim'].values[0])
    dropout = row['dropout'].values[0]
    num_layer_LSTM = int(row['num_layers_LSTM'].values[0])
    num_layer_GRU = int(row['num_layers_GRU'].values[0])

    
    curr_model = LSTMGRU_MLP(input_dim=30, output_dim=1, hidden_dim_1=hidden_1, hidden_dim_2=hidden_2, mlp_dim=mlp_dim, dropout=dropout, num_layers_LSTM=num_layer_LSTM, num_layers_GRU=num_layer_GRU)
    curr_model.load_state_dict(state)
    curr_model.eval()
    
    # Choose proper data to load
    if 'Closeness' in model_name:
        y_test = y_test_closeness
        test_loader = test_closeness
    elif 'Degree' in model_name:
        y_test = y_test_degree
        test_loader = test_degree
    elif 'Betweenness' in model_name:
        y_test = y_test_betweenness
        test_loader = test_betweenness
    elif 'FormanRicci' in model_name:
        y_test = y_test_forman
        test_loader = test_forman
    elif 'Weight' in model_name:
        y_test = y_test_weight
        test_loader = test_weight
    
    print(f'The model: {model_name} has the following results:')
    criterion = nn.BCELoss()
    test_loss, test_aucroc, test_aucpr, test_accuracy = curr_model.test_model_binary(curr_model,test_loader=test_loader, criterion=criterion, y_test=y_test)
    print(f'\tTest Loss: {test_loss}')
    print(f'\tTest AUCROC: {test_aucroc}')
    print(f'\tTest AUCPR: {test_aucpr}')
    print(f'\tTest Accuracy: {test_accuracy}')

Try new seeds to ensure consistency

In [ ]:
runs = []

# Get all names
for model_name in models.keys():
    runs.append(model_name[:model_name.find('_')])
    
runs = np.unique(runs)

df = df[df['run_id'].isin(runs)]
print(df)

In [ ]:
# Display results from dataframe
'''
df = pd.read_csv(csv_file_path)

print('average train loss is: ', df['train_loss'].mean())
print('average valid loss is: ', df['valid_loss'].mean())
print('average train aucroc is: ', df['train_aucroc'].mean())
print('average valid aucroc is: ', df['valid_aucroc'].mean())
print(df.loc[df['valid_aucroc'].idxmax()])
print(df.nlargest(25, 'valid_aucroc'))'''

## Test model parameters, but on different seeds

In [14]:

csv_file_path = 'data/output/results/BinaryTesting/data/cosine_seed_testing.csv'

# Write the header if the file doesn't already exist
if not os.path.isfile(csv_file_path):
    pd.DataFrame(columns=['run_id', 'activation', 'seed', 'hidden_dim_1', 'hidden_dim_2', 'mlp_dim', 'learning_rate', 'dropout', 'l2_regularization', 'num_layers_LSTM', 'num_layers_GRU', 'trained_epochs', 'train_loss', 'valid_loss', 'train_aucroc', 'valid_aucroc', 'train_aucpr', 'valid_aucpr', 'train_accuracy', 'valid_accuracy', 'test_loss', 'test_aucroc', 'test_aucpr', 'test_accuracy']).to_csv(csv_file_path, index=False)

In [ ]:
testing_seeds = [42, 9999, 1, 5555, 1000]

# Test models immediately as we go
wandb.init(
    project="cosine_seed_testing", 
    reinit=True
)

for run in runs:
    row = df[df['run_id'] == str(run)]
    activation = row['activation'].values[0]
    
    if 'Closeness' == activation:
        train_loader = train_closeness
        valid_loader = valid_closeness
        test_loader = test_closeness
        y_train = y_train_closeness
        y_val = y_val_closeness
        y_test = y_test_closeness
    elif 'Degree' == activation:
        train_loader = train_degree
        valid_loader = valid_degree
        test_loader = test_degree
        y_train = y_train_degree
        y_val = y_val_degree
        y_test = y_test_degree
    elif 'Betweenness' == activation:
        train_loader = train_betweenness
        valid_loader = valid_betweenness
        test_loader = test_betweenness
        y_train = y_train_betweenness
        y_val = y_val_betweenness
        y_test = y_test_betweenness
    elif 'FormanRicci' == activation:
        train_loader = train_forman
        valid_loader = valid_forman
        test_loader = test_forman
        y_train = y_train_forman
        y_val = y_val_forman
        y_test = y_test_forman
    elif 'Weight' == activation:
        train_loader = train_weight
        valid_loader = valid_weight
        test_loader = test_weight
        y_train = y_train_weight
        y_val = y_val_weight
        y_test = y_test_weight
    
    # Get hyperparameters:
    hidden_1 = int(row['hidden_dim_1'].values[0])
    hidden_2 = int(row['hidden_dim_2'].values[0])
    mlp_dim = int(row['mlp_dim'].values[0])
    dropout = row['dropout'].values[0]
    l2_val = row['l2_regularization'].values[0]
    lr_val = row['learning_rate'].values[0]
    num_layer_LSTM = int(row['num_layers_LSTM'].values[0])
    num_layer_GRU = int(row['num_layers_GRU'].values[0])
    
        
    for seed in testing_seeds:
        np.random.seed(seed)  # Set the seed here
        
        curr_model = LSTMGRU_MLP(input_dim=30, output_dim=1, hidden_dim_1=hidden_1, hidden_dim_2=hidden_2, mlp_dim=mlp_dim, dropout=dropout, num_layers_LSTM=num_layer_LSTM, num_layers_GRU=num_layer_GRU)
        
        curr_batch_best_aucroc = float('-inf')  # Init

        # Initialize wandb
        run = wandb.init(project="cosine_seed_testing", config={
            'seed': seed,
            'activation': activation,
            'num_layers': num_layer_LSTM,
            'dropout': dropout,
            'l2_regularization': l2_val,
            'hidden_dim_1': hidden_1,
            'hidden_dim_2': hidden_2,
            'mlp_dim': mlp_dim,
            'learning_rate': lr_val,
            'seed': seed
        })

        # Setup
        no_improvement_counter = 0  # Number of epochs that we haven't seen an improvement in the validation AUCROC
        model = LSTMGRU_MLP(input_dim, output_dim, hidden_dim_1=hidden_1, hidden_dim_2=hidden_2, mlp_dim=mlp_dim, dropout=dropout, num_layers_LSTM=num_layer_LSTM, num_layers_GRU=num_layer_GRU)
        optimizer = torch.optim.Adam(model.parameters(), lr=lr_val, weight_decay=l2_val)
        criterion = nn.BCELoss()    

        for epoch in range(num_epochs):
            model.train()
            train_loss, train_aucroc, train_aucpr, train_accuracy = model.train_model_binary(model, train_loader, optimizer, criterion)

            with torch.no_grad():
                model.eval()
                val_preds = []
                valid_loss = 0
                for x, y in valid_loader:
                    output = model(x)  # Maintain hidden state across time steps
                    y = y.squeeze().float()
                    loss = criterion(output, y)
                    valid_loss += loss.item()
                    val_preds.append(output.detach().numpy())

                # Compute metrics
                valid_loss /= len(valid_loader)
                val_preds = np.concatenate(val_preds, axis=0)  # Ensure val_preds is a flat array
                val_preds = np.array(val_preds)
                valid_aucroc = roc_auc_score(y_val, val_preds)
                valid_aucpr = average_precision_score(y_val, val_preds)
                val_pred_labels = [1 if prob >= 0.5 else 0 for prob in val_preds]  # Since accuracy requires labels
                valid_accuracy = accuracy_score(y_val, val_pred_labels)
                
            # Log each epoch results
            wandb.log({
                'epoch': epoch,
                'train_loss': train_loss,
                'valid_loss': valid_loss,
                'train_aucroc': train_aucroc,
                'valid_aucroc': valid_aucroc,
                'train_aucpr': train_aucpr,
                'valid_aucpr': valid_aucpr,
                'train_accuracy': train_accuracy,
                'valid_accuracy': valid_accuracy
            })

            # Optimize for the best aucroc
            if valid_aucroc >= curr_batch_best_aucroc:
                best_model = model
                # Save for dataframe
                best_moment_row = {
                    'run_id': run.name,  # For checking Wandb Logs
                    'activation': activation,
                    'seed': seed,
                    'hidden_dim_1': hidden_1,
                    'hidden_dim_2': hidden_2,
                    'mlp_dim': mlp_dim,
                    'learning_rate': lr_val,
                    'dropout': dropout,
                    'l2_regularization': l2_val,
                    'num_layers_LSTM': num_layer_LSTM,
                    'num_layers_GRU': num_layer_GRU,
                    'trained_epochs': epoch + 1,
                    'train_loss': train_loss,
                    'valid_loss': valid_loss,
                    'train_aucroc': train_aucroc,
                    'valid_aucroc': valid_aucroc,
                    'train_aucpr': train_aucpr,
                    'valid_aucpr': valid_aucpr,
                    'train_accuracy': train_accuracy,
                    'valid_accuracy': valid_accuracy
                }
                
                # Save the model
                curr_batch_best_aucroc = valid_aucroc
                
            
            # Early stopping only after 50 epochs
            if epoch >= 50:
                if valid_aucroc >= curr_batch_best_aucroc:
                    no_improvement_counter = 0
                    curr_batch_best_aucroc = valid_aucroc
                else:
                    no_improvement_counter += 1
                    
                if no_improvement_counter == patience:
                    print(f'Training ending at epoch number: {epoch + 1}')
                    break
                
            # Display current results
            if epoch % 5 - 4 == 0:
                print(f"""
                    Epoch {epoch+1}/{num_epochs}:\n\tTrain Loss: {train_loss}, Validation Loss: {valid_loss}\n\tTrain AUCROC: {train_aucroc}, Validation AUCROC: {valid_aucroc}\n\tTrain AUCPR: {train_aucpr}, Validation AUCPR: {valid_aucpr}\n\tTrain Accuracy: {train_accuracy}, Validation Accuracy: {valid_accuracy}\n
                """)

        test_loss, test_aucroc, test_aucpr, test_accuracy = best_model.test_model_binary(best_model, test_loader, criterion, y_test)

        best_moment_row['test_loss'] = test_loss
        best_moment_row['test_aucroc'] = test_aucroc
        best_moment_row['test_aucpr'] = test_aucpr
        best_moment_row['test_accuracy'] = test_accuracy

        # Save the best moment from this training
        pd.DataFrame([best_moment_row]).to_csv(csv_file_path, mode='a', header=False, index=False)
        
        
wandb.finish()  # Close this round of tests                            

In [16]:

csv_file_path = 'data/output/results/BinaryTesting/data/cosine_best_testing.csv'

# Write the header if the file doesn't already exist
if not os.path.isfile(csv_file_path):
    pd.DataFrame(columns=['run_id', 'activation', 'seed', 'hidden_dim_1', 'hidden_dim_2', 'mlp_dim', 'learning_rate', 'dropout', 'l2_regularization', 'num_layers_LSTM', 'num_layers_GRU', 'trained_epochs', 'train_loss', 'valid_loss', 'train_aucroc', 'valid_aucroc', 'train_aucpr', 'valid_aucpr', 'train_accuracy', 'valid_accuracy', 'test_loss', 'test_aucroc', 'test_aucpr', 'test_accuracy']).to_csv(csv_file_path, index=False)

In [ ]:
# Best models based on different criteria
df = pd.read_csv('data/output/results/BinaryTesting/data/cosine_gs.csv')

top_5_loss = df.nsmallest(25, 'valid_loss')
top_5_aucroc = df.nlargest(25, 'valid_aucroc')
top_5_accuracy = df.nlargest(25, 'valid_accuracy')
top_5_aucpr = df.nlargest(25, 'valid_aucpr')

tmp_df = pd.concat([top_5_loss, top_5_aucroc, top_5_accuracy, top_5_aucpr], axis=0)
print(tmp_df)

# Test models immediately as we go
wandb.init(
    project="cosine_best_testing", 
    reinit=True
)

In [ ]:
# Now test based on parameter results from the csv

for idx, row in tmp_df.iterrows():
    activation = row['activation']
    
    if 'Closeness' == activation:
        train_loader = train_closeness
        valid_loader = valid_closeness
        test_loader = test_closeness
        y_train = y_train_closeness
        y_val = y_val_closeness
        y_test = y_test_closeness
    elif 'Degree' == activation:
        train_loader = train_degree
        valid_loader = valid_degree
        test_loader = test_degree
        y_train = y_train_degree
        y_val = y_val_degree
        y_test = y_test_degree
    elif 'Betweenness' == activation:
        train_loader = train_betweenness
        valid_loader = valid_betweenness
        test_loader = test_betweenness
        y_train = y_train_betweenness
        y_val = y_val_betweenness
        y_test = y_test_betweenness
    elif 'FormanRicci' == activation:
        train_loader = train_forman
        valid_loader = valid_forman
        test_loader = test_forman
        y_train = y_train_forman
        y_val = y_val_forman
        y_test = y_test_forman
    elif 'Weight' == activation:
        train_loader = train_weight
        valid_loader = valid_weight
        test_loader = test_weight
        y_train = y_train_weight
        y_val = y_val_weight
        y_test = y_test_weight
    
    # Get hyperparameters:
    hidden_1 = int(row['hidden_dim_1'])
    hidden_2 = int(row['hidden_dim_2'])
    mlp_dim = int(row['mlp_dim'])
    dropout = row['dropout']
    l2_val = row['l2_regularization']
    lr_val = row['learning_rate']
    num_layer_LSTM = int(row['num_layers_LSTM'])
    num_layer_GRU = int(row['num_layers_GRU'])
    
        
    for seed in testing_seeds:        
        curr_model = LSTMGRU_MLP(input_dim=30, output_dim=1, hidden_dim_1=hidden_1, hidden_dim_2=hidden_2, mlp_dim=mlp_dim, dropout=dropout, num_layers_LSTM=num_layer_LSTM, num_layers_GRU=num_layer_GRU)
        
        curr_batch_best_aucroc = float('-inf')  # Init

        # Initialize wandb
        run = wandb.init(project="cosine_best_testing", config={
            'seed': seed,
            'activation': activation,
            'num_layers': num_layer_LSTM,
            'dropout': dropout,
            'l2_regularization': l2_val,
            'hidden_dim_1': hidden_1,
            'hidden_dim_2': hidden_2,
            'mlp_dim': mlp_dim,
            'learning_rate': lr_val,
            'seed': seed
        })

        # Setup
        no_improvement_counter = 0  # Number of epochs that we haven't seen an improvement in the validation AUCROC
        model = LSTMGRU_MLP(input_dim, output_dim, hidden_dim_1=hidden_1, hidden_dim_2=hidden_2, mlp_dim=mlp_dim, dropout=dropout, num_layers_LSTM=num_layer_LSTM, num_layers_GRU=num_layer_GRU)
        optimizer = torch.optim.Adam(model.parameters(), lr=lr_val, weight_decay=l2_val)
        criterion = nn.BCELoss()    

        for epoch in range(num_epochs):
            model.train()
            train_loss, train_aucroc, train_aucpr, train_accuracy = model.train_model_binary(model, train_loader, optimizer, criterion)

            with torch.no_grad():
                model.eval()
                val_preds = []
                valid_loss = 0
                for x, y in valid_loader:
                    output = model(x)  # Maintain hidden state across time steps
                    y = y.squeeze().float()
                    loss = criterion(output, y)
                    valid_loss += loss.item()
                    val_preds.append(output.detach().numpy())

                # Compute metrics
                valid_loss /= len(valid_loader)
                val_preds = np.concatenate(val_preds, axis=0)  # Ensure val_preds is a flat array
                val_preds = np.array(val_preds)
                valid_aucroc = roc_auc_score(y_val, val_preds)
                valid_aucpr = average_precision_score(y_val, val_preds)
                val_pred_labels = [1 if prob >= 0.5 else 0 for prob in val_preds]  # Since accuracy requires labels
                valid_accuracy = accuracy_score(y_val, val_pred_labels)
                
            # Log each epoch results
            wandb.log({
                'epoch': epoch,
                'train_loss': train_loss,
                'valid_loss': valid_loss,
                'train_aucroc': train_aucroc,
                'valid_aucroc': valid_aucroc,
                'train_aucpr': train_aucpr,
                'valid_aucpr': valid_aucpr,
                'train_accuracy': train_accuracy,
                'valid_accuracy': valid_accuracy
            })

            # Optimize for the best aucroc
            if valid_aucroc >= curr_batch_best_aucroc:
                best_model = model
                # Save for dataframe
                best_moment_row = {
                    'run_id': run.name,  # For checking Wandb Logs
                    'activation': activation,
                    'seed': seed,
                    'hidden_dim_1': hidden_1,
                    'hidden_dim_2': hidden_2,
                    'mlp_dim': mlp_dim,
                    'learning_rate': lr_val,
                    'dropout': dropout,
                    'l2_regularization': l2_val,
                    'num_layers_LSTM': num_layer_LSTM,
                    'num_layers_GRU': num_layer_GRU,
                    'trained_epochs': epoch + 1,
                    'train_loss': train_loss,
                    'valid_loss': valid_loss,
                    'train_aucroc': train_aucroc,
                    'valid_aucroc': valid_aucroc,
                    'train_aucpr': train_aucpr,
                    'valid_aucpr': valid_aucpr,
                    'train_accuracy': train_accuracy,
                    'valid_accuracy': valid_accuracy
                }
                
                # Save the model
                curr_batch_best_aucroc = valid_aucroc
                
            
            # Early stopping only after 50 epochs
            if epoch >= 50:
                if valid_aucroc >= curr_batch_best_aucroc:
                    no_improvement_counter = 0
                    curr_batch_best_aucroc = valid_aucroc
                else:
                    no_improvement_counter += 1
                    
                if no_improvement_counter == patience:
                    print(f'Training ending at epoch number: {epoch + 1}')
                    break
                
            # Display current results
            if epoch % 5 - 4 == 0:
                print(f"""
                    Epoch {epoch+1}/{num_epochs}:\n\tTrain Loss: {train_loss}, Validation Loss: {valid_loss}\n\tTrain AUCROC: {train_aucroc}, Validation AUCROC: {valid_aucroc}\n\tTrain AUCPR: {train_aucpr}, Validation AUCPR: {valid_aucpr}\n\tTrain Accuracy: {train_accuracy}, Validation Accuracy: {valid_accuracy}\n
                """)

        test_loss, test_aucroc, test_aucpr, test_accuracy = best_model.test_model_binary(best_model, test_loader, criterion, y_test)

        best_moment_row['test_loss'] = test_loss
        best_moment_row['test_aucroc'] = test_aucroc
        best_moment_row['test_aucpr'] = test_aucpr
        best_moment_row['test_accuracy'] = test_accuracy

        # Save the best moment from this training
        pd.DataFrame([best_moment_row]).to_csv(csv_file_path, mode='a', header=False, index=False)
        
wandb.finish()

Data Checking

In [23]:
import pandas as pd

pd.set_option('display.max_columns', None)  # Ensure all columns are displayed
pd.set_option('display.width', 1000)  # Ensure rows are displayed in a single line

In [24]:
df = pd.read_csv('data/output/results/BinaryTesting/data/embedding_testing_parallel.csv')
value_counts = df['activation'].value_counts()
print(value_counts)

tmp_df = df[df['activation'] == 'Degree_Forman_Weight']
tmp_df = tmp_df[tmp_df['normalization'] == True]
tmp_df = tmp_df[tmp_df['combo'] == "['LSTM', 'MLP', 'Sigmoid']"]
tmp_df = tmp_df[tmp_df['l2_regularization'] == 1e-05]
print(tmp_df)

activation
Degree_Forman_Weight    6892
Forman_Weight           6729
Degree                  2371
Weight                  1287
Forman                   442
Degree_Forman            425
Degree_Weight            402
Name: count, dtype: int64
                                        run_id        dataset            activation  seed normalization  hidden_size_rnn  hidden_size_other  learning_rate  dropout  l2_regularization  num_layers                       combo  trained_epochs  train_loss  valid_loss  train_aucroc  valid_aucroc  train_aucpr  valid_aucpr  train_accuracy  valid_accuracy  test_loss  test_aucroc  test_aucpr  test_accuracy
9122     networkadex_Degree_Forman_Weight_6152    networkadex  Degree_Forman_Weight  42.0          True             32.0               32.0         0.0001      0.0            0.00001         3.0  ['LSTM', 'MLP', 'Sigmoid']           468.0    0.412090    1.690654      0.884735      0.661765     0.872932     0.613368        0.780488        0.272727   1.073514 

In [25]:
tmp_df = df[df['trained_epochs'] > 10]
tmp_df = tmp_df.nlargest(50, 'valid_aucroc')  # Get the rows with the highest n values

print(tmp_df)

                                        run_id        dataset            activation  seed normalization  hidden_size_rnn  hidden_size_other  learning_rate  dropout  l2_regularization  num_layers                                  combo  trained_epochs  train_loss  valid_loss  train_aucroc  valid_aucroc  train_aucpr  valid_aucpr  train_accuracy  valid_accuracy  test_loss  test_aucroc  test_aucpr  test_accuracy
13355  networkbancor_Degree_Forman_Weight_2330  networkbancor  Degree_Forman_Weight  42.0         False             64.0               32.0         0.0001     0.35            0.00100         3.0              ['LSTM', 'FC', 'Sigmoid']           178.0    0.663092    0.642032      0.599387           1.0     0.649964          1.0        0.534562        0.808511   0.704652     0.365476    0.677462       0.361702
13390  networkbancor_Degree_Forman_Weight_2394  networkbancor  Degree_Forman_Weight  42.0         False             64.0               64.0         0.0001     0.35            0.0

In [26]:
value_counts = tmp_df['combo'].value_counts()
print(value_counts)

value_counts = tmp_df['activation'].value_counts()
print(value_counts)

combo
['GRU', 'Attention', 'FC', 'Sigmoid']    31
['LSTM', 'FC', 'Sigmoid']                 7
['GRU', 'FC', 'Sigmoid']                  4
['LSTM', 'GRU', 'MLP', 'Sigmoid']         3
['GRU', 'MLP', 'Sigmoid']                 3
['LSTM', 'MLP', 'Sigmoid']                2
Name: count, dtype: int64
activation
Forman_Weight           22
Degree_Forman_Weight    20
Degree                   4
Degree_Forman            4
Name: count, dtype: int64


In [27]:
tmp_df = df[df['trained_epochs'] > 0]
tmp_df = tmp_df.nlargest(50, 'test_aucroc')  # Get the rows with the highest n values

print(tmp_df)

                                        run_id        dataset            activation  seed normalization  hidden_size_rnn  hidden_size_other  learning_rate  dropout  l2_regularization  num_layers                                  combo  trained_epochs  train_loss  valid_loss  train_aucroc  valid_aucroc  train_aucpr  valid_aucpr  train_accuracy  valid_accuracy  test_loss  test_aucroc  test_aucpr  test_accuracy
10621  networkaragon_Degree_Forman_Weight_1052  networkaragon  Degree_Forman_Weight  42.0         False             32.0               32.0         0.0001      0.2            0.00100         3.0  ['GRU', 'Attention', 'FC', 'Sigmoid']             3.0    0.695546    0.671728      0.576667      0.841699     0.544572     0.773236        0.489362        0.725490   0.738264     0.895623    0.850694       0.352941
10626  networkaragon_Degree_Forman_Weight_1053  networkaragon  Degree_Forman_Weight  42.0         False             32.0               32.0         0.0001      0.2            0.0

In [28]:
# Figure out what rows would be the best to test on across multiple seeds

# Bad Options:
# Forman only
# Weight only
# Degree + Weight

In [29]:
value_counts = tmp_df['combo'].value_counts()
print(value_counts)

value_counts = tmp_df['activation'].value_counts()
print(value_counts)

value_counts = tmp_df['hidden_size_other'].value_counts()
print(value_counts)


# ['GRU', 'Attention', 'FC', 'Sigmoid']

combo
['GRU', 'Attention', 'FC', 'Sigmoid']    22
['LSTM', 'MLP', 'Sigmoid']               10
['LSTM', 'FC', 'Sigmoid']                 9
['LSTM', 'GRU', 'FC', 'Sigmoid']          5
['LSTM', 'GRU', 'MLP', 'Sigmoid']         4
Name: count, dtype: int64
activation
Degree_Forman_Weight    38
Degree                   8
Forman_Weight            4
Name: count, dtype: int64
hidden_size_other
32.0     17
128.0    14
64.0     13
256.0     6
Name: count, dtype: int64


In [30]:
df = pd.read_csv('data/output/results/BinaryTesting/data/embedding_testing_best_parallel.csv')
tmp_df = df[df['trained_epochs'] > 10]
tmp_df = tmp_df.nlargest(50, 'test_aucroc')  # Get the rows with the highest n values

print(tmp_df)

                                       run_id           dataset            activation   seed  normalization  hidden_size_rnn  hidden_size_other  learning_rate  dropout  l2_regularization  num_layers                                  combo  trained_epochs  train_loss  valid_loss  train_aucroc  valid_aucroc  train_aucpr  valid_aucpr  train_accuracy  valid_accuracy  test_loss  test_aucroc  test_aucpr  test_accuracy
112     networkcentra_Degree_Forman_Weight_17     networkcentra  Degree_Forman_Weight     10           True              128                 32         0.0010      0.0            0.00010           3             ['LSTM', 'MLP', 'Sigmoid']             204    0.588011    0.633608      0.697641      0.927586     0.770156     0.889140        0.576923        0.615385   0.299650     1.000000    1.000000       0.875000
197     networkcentra_Degree_Forman_Weight_77     networkcentra  Degree_Forman_Weight   1234           True              128                 32         0.0010      0.0   

In [31]:
tmp_df = df[df['seed'] == 42]
tmp_df = tmp_df[tmp_df['activation'] == 'Degree_Forman_Weight']
tmp_df = tmp_df[tmp_df['normalization'] == True]
tmp_df = tmp_df[tmp_df['combo'] == "['LSTM', 'GRU', 'MLP', 'Sigmoid']"]
tmp_df = tmp_df[tmp_df['l2_regularization'] == 1e-05]
print(tmp_df)

                                       run_id           dataset            activation   seed  normalization  hidden_size_rnn  hidden_size_other  learning_rate  dropout  l2_regularization  num_layers                       combo  trained_epochs  train_loss  valid_loss  train_aucroc  valid_aucroc  train_aucpr  valid_aucpr  train_accuracy  valid_accuracy  test_loss  test_aucroc  test_aucpr  test_accuracy
13      networkbancor_Degree_Forman_Weight_26     networkbancor  Degree_Forman_Weight  99999           True               32                 64          0.001      0.0            0.00001           3  ['LSTM', 'MLP', 'Sigmoid']              84    0.509363    0.698682      0.816712      0.848485     0.813276     0.683887        0.746544        0.659574   1.401300     0.476190    0.744006       0.680851
127     networkcentra_Degree_Forman_Weight_26     networkcentra  Degree_Forman_Weight  99999           True               32                 64          0.001      0.0            0.00001      